# Multi-Outcome Life Trajectory: Autoregressive Task-Decoder Transformer

This notebook keeps the same preprocessing and task-decoder Transformer modules as `CSCI1470_Parallel_TaskDecoder_Transformer.ipynb`, but predicts future bins sequentially. Training uses teacher forcing; validation and test use autoregressive rollout.

In [ ]:
from pathlib import Path
import json
import math
import random
import time
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


In [ ]:
SEED = 1470

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

PANEL_PATH = Path('../data/processed/psid_panel.csv')
OUT_DIR = Path('../results/autoregressive_task_decoder_transformer_outputs')
OUT_DIR.mkdir(exist_ok=True)

INPUT_BINS = [(20, 21), (22, 23), (24, 25), (26, 27), (28, 29)]
TARGET_BINS = [(30, 31), (32, 33), (34, 35), (36, 37), (38, 39), (40, 41), (42, 43), (44, 45)]

def bin_label(lo, hi):
    return f'a{lo}' if lo == hi else f'a{lo}_{hi}'

input_bins = [bin_label(*b) for b in INPUT_BINS]
target_bins = [bin_label(*b) for b in TARGET_BINS]
print('Input bins:', input_bins)
print('Target bins:', target_bins)

Input bins: ['a20_21', 'a22_23', 'a24_25', 'a26_27', 'a28_29']
Target bins: ['a30_31', 'a32_33', 'a34_35', 'a36_37', 'a38_39', 'a40_41', 'a42_43', 'a44_45']


In [ ]:
panel = pd.read_csv(PANEL_PATH).sort_values(['ID', 'DEMO_AGE_GEN', 'YEAR']).reset_index(drop=True)

# Fresh exact 8:1:1 ID-level split.
ids_all = np.sort(panel['ID'].unique())
rng = np.random.default_rng(SEED)
shuffled = ids_all.copy()
rng.shuffle(shuffled)
n = len(shuffled)
n_train = int(round(0.8 * n))
n_val = int(round(0.1 * n))
split_map = pd.DataFrame({'ID': ids_all, 'split_new': 'test'})
split_map.loc[split_map['ID'].isin(shuffled[:n_train]), 'split_new'] = 'train'
split_map.loc[split_map['ID'].isin(shuffled[n_train:n_train+n_val]), 'split_new'] = 'val'
panel = panel.drop(columns=['split'], errors='ignore').merge(split_map, on='ID', how='left')
panel = panel.rename(columns={'split_new': 'split'})
print('IDs:', panel.ID.nunique())
print(panel[['ID', 'split']].drop_duplicates()['split'].value_counts())

IDs: 2585
split
train    2068
test      259
val       258
Name: count, dtype: int64


In [ ]:
STATIC_CONT = [
    'father_EARN_mean', 'father_FINC_mean', 'father_share_married', 'father_share_divorc',
    'father_EARN_sd', 'father_EDU_MAX', 'father_obs_cnt',
    'mother_EARN_mean', 'mother_FINC_mean', 'mother_share_married', 'mother_share_divorc',
    'mother_EARN_sd', 'mother_EDU_MAX', 'mother_obs_cnt',
]
STATIC_BINARY = ['PAR_BOTH_MISS']
STATIC_CAT = ['DEMO_SEX', 'RACE_ETH_MAJ_COL', 'CGEO_REGION', 'COHORT_BIN']

OUTCOMES = ['EARN_TOT_RDF', 'EMP_WORK', 'FAM_MARSTAT']

def latest_nonmissing(rows, col):
    vals = rows.loc[rows[col].notna()].sort_values(['DEMO_AGE_GEN', 'YEAR'])
    return np.nan if vals.empty else vals.iloc[-1][col]

def aggregate_bin(ego, lo, hi):
    rows = ego[(ego.DEMO_AGE_GEN >= lo) & (ego.DEMO_AGE_GEN <= hi)]
    earn_obs = rows['EARN_TOT_RDF'].notna()
    earn = rows.loc[earn_obs, 'EARN_TOT_RDF'].mean() if earn_obs.any() else np.nan
    emp = latest_nonmissing(rows, 'EMP_WORK')
    mar = latest_nonmissing(rows, 'FAM_MARSTAT')
    return {
        'earn': earn,
        'earn_mask': int(pd.notna(earn)),
        'emp': emp,
        'emp_mask': int(pd.notna(emp)),
        'mar': mar,
        'mar_mask': int(pd.notna(mar)),
    }

rows = []
for ego_id, ego in panel.groupby('ID', sort=True):
    rec = {'ID': ego_id, 'split': ego['split'].iloc[0]}
    for lo, hi in INPUT_BINS:
        label = bin_label(lo, hi)
        agg = aggregate_bin(ego, lo, hi)
        for k, v in agg.items():
            rec[f'in_{label}_{k}'] = v
    for lo, hi in TARGET_BINS:
        label = bin_label(lo, hi)
        agg = aggregate_bin(ego, lo, hi)
        for k, v in agg.items():
            rec[f'tgt_{label}_{k}'] = v
    rows.append(rec)

seq_df = pd.DataFrame(rows)
static_df = panel.sort_values(['ID', 'DEMO_AGE_GEN', 'YEAR']).groupby('ID', as_index=False)[['ID'] + STATIC_CONT + STATIC_BINARY + STATIC_CAT].first()
data = seq_df.merge(static_df, on='ID', how='left')
print(data.shape)
data.head()

(2585, 99)


,ID,split,in_a20_21_earn,in_a20_21_earn_mask,in_a20_21_emp,in_a20_21_emp_mask,in_a20_21_mar,in_a20_21_mar_mask,in_a22_23_earn,in_a22_23_earn_mask,...,mother_share_married,mother_share_divorc,mother_EARN_sd,mother_EDU_MAX,mother_obs_cnt,PAR_BOTH_MISS,DEMO_SEX,RACE_ETH_MAJ_COL,CGEO_REGION,COHORT_BIN
0,4007,val,NaN,0,NaN,0,NaN,0,NaN,0,...,0.222222,0.000000,0.736994,0.0,9.0,0,2,1.0,3.0,0
1,4031,train,NaN,0,NaN,0,NaN,0,0.000000,1,...,1.000000,0.000000,1.101707,1.0,9.0,0,2,1.0,3.0,2
2,4034,train,NaN,0,NaN,0,NaN,0,2.926694,1,...,0.333333,0.666667,2.104212,1.0,9.0,0,1,1.0,3.0,2
3,5003,train,2.393077,1,1.0,1,1.0,1,3.101122,1,...,0.444444,0.000000,0.822426,0.0,9.0,0,1,1.0,3.0,0
4,5004,train,NaN,0,NaN,0,NaN,0,NaN,0,...,0.777778,0.000000,0.832096,0.0,9.0,0,2,1.0,3.0,1


In [ ]:
train_mask = data['split'].eq('train')

# Category maps are fit on training data only. 0=missing, 1=unseen category.
def fit_cat_map(series):
    vals = sorted(series.dropna().astype(float).unique().tolist())
    return {str(v): i + 2 for i, v in enumerate(vals)}

def encode_with_map(series, mapping):
    def enc(v):
        if pd.isna(v):
            return 0
        return mapping.get(str(float(v)), 1)
    return series.map(enc).astype('int64')

emp_values = pd.concat([data.loc[train_mask, f'in_{b}_emp'] for b in input_bins] + [data.loc[train_mask, f'tgt_{b}_emp'] for b in target_bins])
mar_values = pd.concat([data.loc[train_mask, f'in_{b}_mar'] for b in input_bins] + [data.loc[train_mask, f'tgt_{b}_mar'] for b in target_bins])
emp_map = fit_cat_map(emp_values)
mar_map = fit_cat_map(mar_values)
static_cat_maps = {c: fit_cat_map(data.loc[train_mask, c]) for c in STATIC_CAT}

print('emp classes:', emp_map)
print('mar classes:', mar_map)
print('static cat maps:', {k: len(v) for k, v in static_cat_maps.items()})

emp classes: {'0.0': 2, '1.0': 3}
mar classes: {'1.0': 2, '2.0': 3, '3.0': 4, '4.0': 5, '5.0': 6}
static cat maps: {'DEMO_SEX': 2, 'RACE_ETH_MAJ_COL': 4, 'CGEO_REGION': 5, 'COHORT_BIN': 3}


In [ ]:
# Scale continuous inputs using train only.
earn_input_cols = [f'in_{b}_earn' for b in input_bins]
static_cont_cols = STATIC_CONT.copy()

earn_mean = data.loc[train_mask, earn_input_cols].stack().mean()
earn_std = data.loc[train_mask, earn_input_cols].stack().std(ddof=0)
earn_std = 1.0 if not np.isfinite(earn_std) or earn_std < 1e-8 else earn_std

static_stats = {}
for c in static_cont_cols:
    s = data.loc[train_mask, c].dropna().astype(float)
    mu = s.mean() if len(s) else 0.0
    sd = s.std(ddof=0) if len(s) else 1.0
    sd = 1.0 if not np.isfinite(sd) or sd < 1e-8 else sd
    static_stats[c] = (mu, sd)

ids = data['ID'].to_numpy(dtype=np.int64)
split = data['split'].astype(str).to_numpy()
N = len(data)
Tin = len(input_bins)
Tout = len(target_bins)

# History token continuous: scaled earnings plus masks for all three observed outcomes.
hist_cont = np.zeros((N, Tin, 4), dtype=np.float32)
hist_emp = np.zeros((N, Tin), dtype=np.int64)
hist_mar = np.zeros((N, Tin), dtype=np.int64)
for t, b in enumerate(input_bins):
    earn = ((data[f'in_{b}_earn'].astype(float) - earn_mean) / earn_std).fillna(0.0)
    hist_cont[:, t, 0] = earn.to_numpy(np.float32)
    hist_cont[:, t, 1] = data[f'in_{b}_earn_mask'].fillna(0).to_numpy(np.float32)
    hist_cont[:, t, 2] = data[f'in_{b}_emp_mask'].fillna(0).to_numpy(np.float32)
    hist_cont[:, t, 3] = data[f'in_{b}_mar_mask'].fillna(0).to_numpy(np.float32)
    hist_emp[:, t] = encode_with_map(data[f'in_{b}_emp'], emp_map).to_numpy()
    hist_mar[:, t] = encode_with_map(data[f'in_{b}_mar'], mar_map).to_numpy()

static_cont = np.zeros((N, len(static_cont_cols) + len(STATIC_BINARY)), dtype=np.float32)
for j, c in enumerate(static_cont_cols):
    mu, sd = static_stats[c]
    static_cont[:, j] = ((data[c].astype(float) - mu) / sd).fillna(0.0).to_numpy(np.float32)
for j, c in enumerate(STATIC_BINARY, start=len(static_cont_cols)):
    static_cont[:, j] = data[c].fillna(0).to_numpy(np.float32)

static_cat = np.stack([encode_with_map(data[c], static_cat_maps[c]).to_numpy() for c in STATIC_CAT], axis=1).astype(np.int64)

# Targets. Earnings stays on original signed-log scale. Categorical targets are class ids: -1 means missing.
y_earn = np.zeros((N, Tout), dtype=np.float32)
y_emp = np.full((N, Tout), -1, dtype=np.int64)
y_mar = np.full((N, Tout), -1, dtype=np.int64)
mask_earn = np.zeros((N, Tout), dtype=np.float32)
mask_emp = np.zeros((N, Tout), dtype=np.float32)
mask_mar = np.zeros((N, Tout), dtype=np.float32)
for t, b in enumerate(target_bins):
    earn = data[f'tgt_{b}_earn']
    y_earn[:, t] = earn.fillna(0.0).to_numpy(np.float32)
    mask_earn[:, t] = data[f'tgt_{b}_earn_mask'].fillna(0).to_numpy(np.float32)
    emp_enc = encode_with_map(data[f'tgt_{b}_emp'], emp_map).to_numpy()
    mar_enc = encode_with_map(data[f'tgt_{b}_mar'], mar_map).to_numpy()
    emp_m = data[f'tgt_{b}_emp_mask'].fillna(0).to_numpy(np.float32)
    mar_m = data[f'tgt_{b}_mar_mask'].fillna(0).to_numpy(np.float32)
    y_emp[:, t] = np.where(emp_m > 0, emp_enc, -1)
    y_mar[:, t] = np.where(mar_m > 0, mar_enc, -1)
    mask_emp[:, t] = emp_m
    mask_mar[:, t] = mar_m

print('hist_cont', hist_cont.shape, 'hist_emp', hist_emp.shape, 'static_cont', static_cont.shape)
print('target masks earn/emp/mar:', mask_earn.sum(axis=0).astype(int).tolist(), mask_emp.sum(axis=0).astype(int).tolist(), mask_mar.sum(axis=0).astype(int).tolist())

hist_cont (2585, 5, 4) hist_emp (2585, 5) static_cont (2585, 15)
target masks earn/emp/mar: [2506, 2522, 2541, 2513, 2517, 2519, 2549, 2493] [2510, 2526, 2541, 2516, 2515, 2520, 2549, 2489] [2511, 2526, 2543, 2518, 2519, 2524, 2550, 2494]


In [ ]:
class MultiOutcomeDataset(Dataset):
    def __init__(self, split_name):
        idx = np.where(split == split_name)[0]
        self.ids = torch.from_numpy(ids[idx])
        self.hist_cont = torch.from_numpy(hist_cont[idx])
        self.hist_emp = torch.from_numpy(hist_emp[idx])
        self.hist_mar = torch.from_numpy(hist_mar[idx])
        self.static_cont = torch.from_numpy(static_cont[idx])
        self.static_cat = torch.from_numpy(static_cat[idx])
        self.y_earn = torch.from_numpy(y_earn[idx])
        self.y_emp = torch.from_numpy(y_emp[idx])
        self.y_mar = torch.from_numpy(y_mar[idx])
        self.mask_earn = torch.from_numpy(mask_earn[idx])
        self.mask_emp = torch.from_numpy(mask_emp[idx])
        self.mask_mar = torch.from_numpy(mask_mar[idx])

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        return {k: getattr(self, k)[i] for k in [
            'ids', 'hist_cont', 'hist_emp', 'hist_mar', 'static_cont', 'static_cat',
            'y_earn', 'y_emp', 'y_mar', 'mask_earn', 'mask_emp', 'mask_mar'
        ]}

def make_loader(split_name, batch_size=128, shuffle=False):
    return DataLoader(MultiOutcomeDataset(split_name), batch_size=batch_size, shuffle=shuffle)

print(len(MultiOutcomeDataset('train')), len(MultiOutcomeDataset('val')), len(MultiOutcomeDataset('test')))

2068 258 259


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class StaticTokenizer(nn.Module):
    def __init__(self, n_cont, cat_vocab_sizes, d_model, cat_emb_dim=12):
        super().__init__()
        self.cont_feature = nn.Parameter(torch.randn(n_cont, d_model) * 0.02)
        self.cont_value = nn.Linear(1, d_model)
        self.cat_embs = nn.ModuleList([nn.Embedding(v, cat_emb_dim, padding_idx=0) for v in cat_vocab_sizes])
        self.cat_proj = nn.ModuleList([nn.Linear(cat_emb_dim, d_model) for _ in cat_vocab_sizes])
        self.cat_feature = nn.Parameter(torch.randn(len(cat_vocab_sizes), d_model) * 0.02)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, cont, cat):
        toks = [self.cont_value(cont.unsqueeze(-1)) + self.cont_feature.unsqueeze(0)]
        for j, (emb, proj) in enumerate(zip(self.cat_embs, self.cat_proj)):
            toks.append((proj(emb(cat[:, j])) + self.cat_feature[j].unsqueeze(0)).unsqueeze(1))
        return self.norm(torch.cat(toks, dim=1))

def earn_to_scaled(x):
    return (x - float(earn_mean)) / float(earn_std)

def earn_from_scaled(x):
    return x * float(earn_std) + float(earn_mean)

def move_batch(batch):
    return {k: v.to(DEVICE) if torch.is_tensor(v) and k != 'ids' else v for k, v in batch.items()}

def masked_mse_scaled(pred_scaled, y_raw, mask):
    valid = mask > 0
    if not valid.any():
        return pred_scaled.sum() * 0.0
    y_scaled = earn_to_scaled(y_raw)
    return F.mse_loss(pred_scaled[valid], y_scaled[valid])

def masked_ce(logits, y, mask):
    valid = (mask > 0) & (y >= 0)
    if not valid.any():
        return logits.sum() * 0.0
    return F.cross_entropy(logits[valid], y[valid])

LOSS_WEIGHTS = {'earn': 1.0, 'emp': 0.5, 'mar': 0.5}

def multitask_loss(earn_scaled, emp, mar, batch):
    return (LOSS_WEIGHTS['earn'] * masked_mse_scaled(earn_scaled, batch['y_earn'], batch['mask_earn']) +
            LOSS_WEIGHTS['emp'] * masked_ce(emp, batch['y_emp'], batch['mask_emp']) +
            LOSS_WEIGHTS['mar'] * masked_ce(mar, batch['y_mar'], batch['mask_mar']))


## Autoregressive Task-Decoder Transformer

The encoder, static tokenizer, cross-attention decoder block, and task-specific decoder heads match the task-decoder Transformer. The change is procedural: after each predicted target bin, the predicted or teacher-forced outcomes are appended as the next history token.

In [ ]:
class CrossDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.15, ff_mult=4):
        super().__init__()
        self.hist_cross = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.static_cross = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_mult*d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_mult*d_model, d_model)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, q, hist_mem, static_mem):
        y, _ = self.hist_cross(q, hist_mem, hist_mem, need_weights=False)
        q = self.norm1(q + self.drop(y))
        y, _ = self.static_cross(q, static_mem, static_mem, need_weights=False)
        q = self.norm2(q + self.drop(y))
        q = self.norm3(q + self.drop(self.ff(q)))
        return q

class StepTaskSpecificDecoder(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.query = nn.Parameter(torch.randn(Tout, d_model) * 0.02)
        self.blocks = nn.ModuleList([CrossDecoderBlock(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])

    def forward(self, hist_mem, static_mem, t_idx):
        q = self.query[t_idx].view(1, 1, -1).expand(hist_mem.size(0), 1, -1)
        for block in self.blocks:
            q = block(q, hist_mem, static_mem)
        return q.squeeze(1)

class MultiOutcomeAutoregressiveTaskDecoderTransformer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_hist_layers=1, n_task_layers=1, dropout=0.25,
                 emp_emb_dim=8, mar_emb_dim=8, static_cat_emb_dim=12):
        super().__init__()
        self.n_emp = max(emp_map.values()) + 1
        self.n_mar = max(mar_map.values()) + 1
        static_cat_vocab_sizes = [max(m.values()) + 1 for m in static_cat_maps.values()]
        self.hist_emp_emb = nn.Embedding(self.n_emp, emp_emb_dim, padding_idx=0)
        self.hist_mar_emb = nn.Embedding(self.n_mar, mar_emb_dim, padding_idx=0)
        self.hist_in = nn.Linear(4 + emp_emb_dim + mar_emb_dim, d_model)
        self.hist_pos = PositionalEncoding(d_model, max_len=16)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.hist_encoder = nn.TransformerEncoder(enc_layer, num_layers=n_hist_layers)
        self.static_tok = StaticTokenizer(static_cont.shape[1], static_cat_vocab_sizes, d_model, static_cat_emb_dim)
        self.earn_dec = StepTaskSpecificDecoder(d_model, n_heads, n_task_layers, dropout)
        self.emp_dec = StepTaskSpecificDecoder(d_model, n_heads, n_task_layers, dropout)
        self.mar_dec = StepTaskSpecificDecoder(d_model, n_heads, n_task_layers, dropout)
        self.earn_head_scaled = nn.Linear(d_model, 1)
        self.emp_head = nn.Linear(d_model, self.n_emp)
        self.mar_head = nn.Linear(d_model, self.n_mar)

    def encode(self, hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b):
        h = torch.cat([hist_cont_b, self.hist_emp_emb(hist_emp_b), self.hist_mar_emb(hist_mar_b)], dim=-1)
        h = self.hist_pos(self.hist_in(h))
        hist_mem = self.hist_encoder(h)
        static_mem = self.static_tok(static_cont_b, static_cat_b)
        return hist_mem, static_mem

    def predict_step(self, hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b, t_idx):
        hist_mem, static_mem = self.encode(hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b)
        earn_h = self.earn_dec(hist_mem, static_mem, t_idx)
        emp_h = self.emp_dec(hist_mem, static_mem, t_idx)
        mar_h = self.mar_dec(hist_mem, static_mem, t_idx)
        return self.earn_head_scaled(earn_h).squeeze(-1), self.emp_head(emp_h), self.mar_head(mar_h)

    def _next_token(self, earn_scaled_t, emp_logits_t, mar_logits_t, batch, t_idx, teacher_forcing_ratio=1.0):
        B = earn_scaled_t.size(0)
        device = earn_scaled_t.device
        pred_earn = earn_scaled_t.detach()
        pred_emp = emp_logits_t.argmax(-1).detach()
        pred_mar = mar_logits_t.argmax(-1).detach()

        true_earn = earn_to_scaled(batch['y_earn'][:, t_idx]).to(device)
        true_emp = batch['y_emp'][:, t_idx].to(device)
        true_mar = batch['y_mar'][:, t_idx].to(device)

        earn_obs = batch['mask_earn'][:, t_idx].to(device) > 0
        emp_obs = (batch['mask_emp'][:, t_idx].to(device) > 0) & (true_emp >= 0)
        mar_obs = (batch['mask_mar'][:, t_idx].to(device) > 0) & (true_mar >= 0)

        if teacher_forcing_ratio >= 1.0:
            use_true_earn = earn_obs
            use_true_emp = emp_obs
            use_true_mar = mar_obs
        elif teacher_forcing_ratio <= 0.0:
            use_true_earn = torch.zeros(B, dtype=torch.bool, device=device)
            use_true_emp = torch.zeros(B, dtype=torch.bool, device=device)
            use_true_mar = torch.zeros(B, dtype=torch.bool, device=device)
        else:
            draw = torch.rand(B, device=device)
            use_true_earn = earn_obs & (draw < teacher_forcing_ratio)
            draw = torch.rand(B, device=device)
            use_true_emp = emp_obs & (draw < teacher_forcing_ratio)
            draw = torch.rand(B, device=device)
            use_true_mar = mar_obs & (draw < teacher_forcing_ratio)

        next_earn = torch.where(use_true_earn, true_earn, pred_earn)
        next_emp = torch.where(use_true_emp, true_emp, pred_emp).clamp(min=0, max=self.n_emp - 1)
        next_mar = torch.where(use_true_mar, true_mar, pred_mar).clamp(min=0, max=self.n_mar - 1)

        # These mask channels mean the previous generated token is available to the decoder.
        step_cont = torch.ones(B, 4, dtype=batch['hist_cont'].dtype, device=device)
        step_cont[:, 0] = next_earn
        return step_cont, next_emp.long(), next_mar.long()

    def forward(self, hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b, batch=None, teacher_forcing_ratio=1.0):
        hist_cont_cur = hist_cont_b
        hist_emp_cur = hist_emp_b
        hist_mar_cur = hist_mar_b
        earn_steps, emp_steps, mar_steps = [], [], []
        for t_idx in range(Tout):
            earn_t, emp_t, mar_t = self.predict_step(hist_cont_cur, hist_emp_cur, hist_mar_cur, static_cont_b, static_cat_b, t_idx)
            earn_steps.append(earn_t)
            emp_steps.append(emp_t)
            mar_steps.append(mar_t)
            if t_idx < Tout - 1:
                if batch is None:
                    step_cont = torch.ones(hist_cont_cur.size(0), 4, dtype=hist_cont_cur.dtype, device=hist_cont_cur.device)
                    step_cont[:, 0] = earn_t.detach()
                    step_emp = emp_t.argmax(-1).detach()
                    step_mar = mar_t.argmax(-1).detach()
                else:
                    step_cont, step_emp, step_mar = self._next_token(earn_t, emp_t, mar_t, batch, t_idx, teacher_forcing_ratio)
                hist_cont_cur = torch.cat([hist_cont_cur, step_cont.unsqueeze(1)], dim=1)
                hist_emp_cur = torch.cat([hist_emp_cur, step_emp.unsqueeze(1)], dim=1)
                hist_mar_cur = torch.cat([hist_mar_cur, step_mar.unsqueeze(1)], dim=1)
        return torch.stack(earn_steps, dim=1), torch.stack(emp_steps, dim=1), torch.stack(mar_steps, dim=1)

    @torch.no_grad()
    def rollout(self, hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b):
        return self.forward(hist_cont_b, hist_emp_b, hist_mar_b, static_cont_b, static_cat_b, batch=None, teacher_forcing_ratio=0.0)


In [ ]:
def autoregressive_loss_fn(model, batch, teacher_forcing_ratio=1.0):
    earn_scaled, emp, mar = model(
        batch['hist_cont'], batch['hist_emp'], batch['hist_mar'],
        batch['static_cont'], batch['static_cat'],
        batch=batch, teacher_forcing_ratio=teacher_forcing_ratio,
    )
    return multitask_loss(earn_scaled, emp, mar, batch)

def autoregressive_val_earn_mse(model, loader):
    model.eval()
    total, nobs = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            earn_scaled, _, _ = model.rollout(batch['hist_cont'], batch['hist_emp'], batch['hist_mar'], batch['static_cont'], batch['static_cat'])
            valid = batch['mask_earn'] > 0
            if valid.any():
                y_scaled = earn_to_scaled(batch['y_earn'])
                total += F.mse_loss(earn_scaled[valid], y_scaled[valid], reduction='sum').item()
                nobs += int(valid.sum().item())
    return total / max(nobs, 1)

def evaluate_autoregressive_transformer(model, split_name, model_name, batch_size=256):
    loader = make_loader(split_name, batch_size)
    model.eval()
    rows=[]; pred_rows=[]
    with torch.no_grad():
        for batch in loader:
            ids_b = batch['ids'].numpy()
            batch = move_batch(batch)
            earn_scaled, emp_logits, mar_logits = model.rollout(batch['hist_cont'], batch['hist_emp'], batch['hist_mar'], batch['static_cont'], batch['static_cat'])
            pred_earn = earn_from_scaled(earn_scaled).cpu().numpy()
            pred_emp = emp_logits.argmax(-1).cpu().numpy()
            pred_mar = mar_logits.argmax(-1).cpu().numpy()
            y_e = batch['y_earn'].cpu().numpy(); y_emp_b = batch['y_emp'].cpu().numpy(); y_mar_b = batch['y_mar'].cpu().numpy()
            m_e = batch['mask_earn'].cpu().numpy(); m_emp_b = batch['mask_emp'].cpu().numpy(); m_mar_b = batch['mask_mar'].cpu().numpy()
            for j,b in enumerate(target_bins):
                for i,ego_id in enumerate(ids_b):
                    pred_rows.append({'ID':int(ego_id),'split':split_name,'target_bin':b,
                                      'pred_earn':pred_earn[i,j],'true_earn':y_e[i,j],'mask_earn':m_e[i,j],
                                      'pred_emp':pred_emp[i,j],'true_emp':y_emp_b[i,j],'mask_emp':m_emp_b[i,j],
                                      'pred_mar':pred_mar[i,j],'true_mar':y_mar_b[i,j],'mask_mar':m_mar_b[i,j]})
    pred_df = pd.DataFrame(pred_rows)
    for b in target_bins:
        sub = pred_df[pred_df.target_bin.eq(b)]
        valid = sub.mask_earn > 0
        if valid.sum() > 1:
            y=sub.loc[valid,'true_earn']; p=sub.loc[valid,'pred_earn']; sp=spearmanr(y,p).correlation
            rows.append({'model':model_name,'split':split_name,'target_bin':b,'target':'earn','n':int(valid.sum()),
                         'rmse':float(np.sqrt(mean_squared_error(y,p))),'mae':float(mean_absolute_error(y,p)),
                         'r2':float(r2_score(y,p)),'spearman':float(sp) if np.isfinite(sp) else np.nan})
        for target, pred_col, true_col, mask_col in [('emp','pred_emp','true_emp','mask_emp'),('mar','pred_mar','true_mar','mask_mar')]:
            valid = sub[mask_col] > 0
            if valid.sum() > 1:
                rows.append({'model':model_name,'split':split_name,'target_bin':b,'target':target,'n':int(valid.sum()),
                             'accuracy':float(accuracy_score(sub.loc[valid,true_col], sub.loc[valid,pred_col])),
                             'f1_macro':float(f1_score(sub.loc[valid,true_col], sub.loc[valid,pred_col], average='macro', zero_division=0))})
    return pd.DataFrame(rows), pred_df


In [ ]:
def train_autoregressive_model(cfg, epochs=60, patience=7):
    set_seed(cfg.get('seed', SEED))
    model_cfg = {k:v for k,v in cfg.items() if k not in ['lr','weight_decay','batch_size','seed']}
    model = MultiOutcomeAutoregressiveTaskDecoderTransformer(**model_cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    train_loader = make_loader('train', cfg['batch_size'], shuffle=True)
    val_loader = make_loader('val', cfg['batch_size'])
    best_val, bad, best_epoch = float('inf'), 0, 0
    best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    hist=[]
    for ep in range(epochs):
        model.train(); tr=0.0; ntr=0
        for batch in train_loader:
            batch = move_batch(batch)
            opt.zero_grad()
            loss = autoregressive_loss_fn(model, batch, teacher_forcing_ratio=1.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tr += loss.item()*batch['hist_cont'].size(0); ntr += batch['hist_cont'].size(0)
        sched.step(); tr/=max(ntr,1)
        val_mse = autoregressive_val_earn_mse(model, val_loader)
        hist.append({'epoch':ep, 'train_loss':tr, 'val_earn_mse':val_mse, 'teacher_forcing_ratio':1.0})
        print(f'epoch {ep:03d} train {tr:.4f} val_earn_mse {val_mse:.4f}')
        if val_mse < best_val - 1e-5:
            best_val, bad, best_epoch = val_mse, 0, ep
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, best_epoch, pd.DataFrame(hist)


In [ ]:
ORIGINAL_OUT_DIR = Path('../results/multioutcome_transformer_lasso_outputs')
ORIGINAL_SEARCH_PATH = ORIGINAL_OUT_DIR / 'task_decoder_transformer_search.csv'

fallback_cfg = {
    'd_model': 64,
    'n_heads': 4,
    'n_hist_layers': 1,
    'n_task_layers': 1,
    'dropout': 0.25,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 128,
    'seed': SEED,
}

if ORIGINAL_SEARCH_PATH.exists():
    original_search = pd.read_csv(ORIGINAL_SEARCH_PATH).sort_values('val_earn_mse')
    best_row = original_search.iloc[0]
    ar_cfg = {k: fallback_cfg[k] for k in fallback_cfg}
    for k in ['d_model', 'n_heads', 'n_hist_layers', 'n_task_layers', 'batch_size', 'seed']:
        ar_cfg[k] = int(best_row[k])
    for k in ['dropout', 'lr', 'weight_decay']:
        ar_cfg[k] = float(best_row[k])
    print('Using original task-decoder best hyperparameters:')
else:
    ar_cfg = fallback_cfg.copy()
    print('Original search file not found; using fallback hyperparameters:')
print(ar_cfg)

model, val_mse, best_epoch, history = train_autoregressive_model(ar_cfg, epochs=60, patience=7)
best_ar = {'val_earn_mse': val_mse, 'cfg': ar_cfg.copy(), 'model': model, 'history': history.copy(), 'best_epoch': best_epoch}

torch.save({'state_dict': model.state_dict(), 'cfg': ar_cfg, 'target_bins': target_bins}, OUT_DIR/'best_autoregressive_task_decoder_transformer.pt')
pd.DataFrame([dict(ar_cfg, val_earn_mse=val_mse, best_epoch=best_epoch)]).to_csv(OUT_DIR/'autoregressive_task_decoder_transformer_search.csv', index=False)
history.to_csv(OUT_DIR/'autoregressive_task_decoder_transformer_best_history.csv', index=False)
pd.DataFrame([dict(ar_cfg, val_earn_mse=val_mse, best_epoch=best_epoch)])


Using original task-decoder best hyperparameters:
{'d_model': 64, 'n_heads': 4, 'n_hist_layers': 1, 'n_task_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.001, 'batch_size': 128, 'seed': 1470}
epoch 000 train 1.7319 val_earn_mse 1.0386
epoch 001 train 1.1507 val_earn_mse 1.1537
epoch 002 train 1.0190 val_earn_mse 0.9743
epoch 003 train 0.9556 val_earn_mse 1.1569
epoch 004 train 0.8934 val_earn_mse 1.0481
epoch 005 train 0.8407 val_earn_mse 1.0903
epoch 006 train 0.8392 val_earn_mse 1.1066
epoch 007 train 0.8105 val_earn_mse 1.0181
epoch 008 train 0.8032 val_earn_mse 1.0013
epoch 009 train 0.7982 val_earn_mse 0.9617
epoch 010 train 0.7971 val_earn_mse 1.1351
epoch 011 train 0.7895 val_earn_mse 1.1071
epoch 012 train 0.7906 val_earn_mse 1.1355
epoch 013 train 0.7812 val_earn_mse 1.1098
epoch 014 train 0.7781 val_earn_mse 1.0679
epoch 015 train 0.7752 val_earn_mse 1.0838
epoch 016 train 0.7748 val_earn_mse 1.0492


,d_model,n_heads,n_hist_layers,n_task_layers,dropout,lr,weight_decay,batch_size,seed,val_earn_mse,best_epoch
0,64,4,1,1,0.1,0.001,0.001,128,1470,0.96173,9


In [ ]:
ar_val_metrics, ar_val_preds = evaluate_autoregressive_transformer(best_ar['model'], 'val', 'AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER')
ar_test_metrics, ar_test_preds = evaluate_autoregressive_transformer(best_ar['model'], 'test', 'AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER')
ar_metrics = pd.concat([ar_val_metrics, ar_test_metrics], ignore_index=True)
ar_preds = pd.concat([ar_val_preds, ar_test_preds], ignore_index=True)
ar_metrics.to_csv(OUT_DIR/'autoregressive_task_decoder_transformer_metrics.csv', index=False)
ar_preds.to_csv(OUT_DIR/'autoregressive_task_decoder_transformer_predictions.csv', index=False)
ar_metrics


,model,split,target_bin,target,n,rmse,mae,r2,spearman,accuracy,f1_macro
0,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a30_31,earn,248,0.890936,0.578190,0.563111,0.758588,NaN,NaN
1,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a30_31,emp,250,NaN,NaN,NaN,NaN,0.840000,0.708251
2,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a30_31,mar,250,NaN,NaN,NaN,NaN,0.880000,0.738193
3,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a32_33,earn,251,1.019380,0.739546,0.385319,0.662735,NaN,NaN
4,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a32_33,emp,251,NaN,NaN,NaN,NaN,0.844622,0.680108
5,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a32_33,mar,251,NaN,NaN,NaN,NaN,0.792829,0.432164
6,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a34_35,earn,257,1.071435,0.773610,0.300861,0.652968,NaN,NaN
7,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a34_35,emp,257,NaN,NaN,NaN,NaN,0.824903,0.623351
8,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a34_35,mar,257,NaN,NaN,NaN,NaN,0.782101,0.566996
9,AUTOREGRESSIVE_TASK_DECODER_TRANSFORMER,val,a36_37,earn,252,1.135986,0.822852,0.217692,0.617204,NaN,NaN
